# What is a British racecourse?

Study 03 separates **racecourse identity**, **stable physical course/track identity**, and lower-level **route/configuration/characteristic**. The 60 individual racecourse notebooks are the source of truth for the national consolidation. `candidate_course_label` remains a source classification rather than a governed venue or track ID.

In [1]:
from pathlib import Path
import ast
import json
import pandas as pd

# Resolve repository root safely from any normal notebook launch location.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'studies').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RACECOURSE_DIR = PROJECT_ROOT / 'studies' / 'jurisdictions' / 'great_britain' / 'racecourses'
assert RACECOURSE_DIR.exists(), f'Racecourse notebook directory not found: {RACECOURSE_DIR}'

racecourse_notebooks = sorted(RACECOURSE_DIR.glob('*.ipynb'))
assert len(racecourse_notebooks) == 60, f'Expected 60 racecourse notebooks, found {len(racecourse_notebooks)}'
print('racecourse notebooks:', len(racecourse_notebooks))

racecourse notebooks: 60


In [2]:
def assigned_names(source):
    tree = ast.parse(source)
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if isinstance(target, ast.Name):
                    names.add(target.id)
    return names


def extract_dataframe(notebook_path, variable_name, *, required=True):
    """Extract a self-contained DataFrame assignment from a venue notebook."""
    notebook = json.loads(notebook_path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        if variable_name not in names:
            continue
        namespace = {'pd': pd}
        exec(source, namespace)
        value = namespace.get(variable_name)
        if isinstance(value, pd.DataFrame):
            return value.copy()
    if required:
        raise RuntimeError(f'{variable_name!r} not found as a DataFrame in {notebook_path.name}')
    return pd.DataFrame()

## Racecourse identity and source labels

In [3]:
source_label_mapping = pd.concat(
    [extract_dataframe(path, 'source_label_mapping') for path in racecourse_notebooks],
    ignore_index=True, sort=False,
)

print('source labels:', source_label_mapping['candidate_course_label'].nunique())
print('governed racecourse identities:', source_label_mapping['racecourse_identity'].nunique())

assert source_label_mapping['candidate_course_label'].nunique() == 65
assert source_label_mapping['racecourse_identity'].nunique() == 60

source labels: 65
governed racecourse identities: 60


The 65 British source labels consolidate to **60 governed racecourse identities**. This directly shows why `COUNT(DISTINCT candidate_course_label)` is not a racecourse count: labels such as Kempton/Kempton (AW), Lingfield/Lingfield (AW), Newcastle/Newcastle (AW), Newmarket/Newmarket (July), and Southwell/Southwell (AW) preserve useful source distinctions inside a smaller set of racecourse identities.

## Course/track inventory

In [4]:
course_inventory = pd.concat(
    [extract_dataframe(path, 'course_inventory') for path in racecourse_notebooks],
    ignore_index=True, sort=False,
)

print('racecourses:', course_inventory['racecourse_identity'].nunique())
print('course/track inventory records:', len(course_inventory))

assert course_inventory['racecourse_identity'].nunique() == 60
assert len(course_inventory) == 90

racecourses: 60
course/track inventory records: 90


## Stable course/track identities

Inventory rows are not automatically stable identities. Southwell's Fibresand/Tapeta rows are successive surface states of one all-weather track; Newcastle's former turf Flat track and Tapeta track are successive states of one persistent Flat-track identity; Windsor's dated Jump rows are temporary configurations of its turf course. These are the only national consolidation collapses applied here.

In [5]:
stable_course_inventory = course_inventory.copy()
stable_course_inventory['stable_course_identity'] = stable_course_inventory['course_or_track_name']

resolved_collapses = {
    ('Southwell', 'All-Weather Flat Track — Fibresand'): 'All-Weather Flat Track',
    ('Southwell', 'All-Weather Flat Track — Tapeta'): 'All-Weather Flat Track',
    ('Newcastle', 'Former Flat Turf Track'): 'Flat Track',
    ('Newcastle', 'All-Weather Tapeta Track'): 'Flat Track',
    ('Windsor', 'Traditional Figure-of-Eight Turf Course'): 'Windsor Turf Course',
    ('Windsor', '2024/25 Jump Extended Left-Hand Oval'): 'Windsor Turf Course',
    ('Windsor', '2025/26 Jump Figure-of-Eight Configuration'): 'Windsor Turf Course',
}

for (racecourse, raw_name), stable_name in resolved_collapses.items():
    mask = (
        stable_course_inventory['racecourse_identity'].eq(racecourse)
        & stable_course_inventory['course_or_track_name'].eq(raw_name)
    )
    stable_course_inventory.loc[mask, 'stable_course_identity'] = stable_name

stable_identities = (
    stable_course_inventory[['racecourse_identity', 'stable_course_identity']]
    .drop_duplicates()
    .sort_values(['racecourse_identity', 'stable_course_identity'])
    .reset_index(drop=True)
)

identity_counts = (
    stable_identities.groupby('racecourse_identity').size()
    .rename('stable_course_identities').reset_index()
)

print('inventory records:', len(course_inventory))
print('stable course/track identities:', len(stable_identities))
print('racecourses with multiple stable identities:', (identity_counts['stable_course_identities'] > 1).sum())
print(identity_counts['stable_course_identities'].value_counts().sort_index())

assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 20
stable_identities

inventory records: 90
stable course/track identities: 86
racecourses with multiple stable identities: 20
stable_course_identities
1    40
2    15
3     4
4     1
Name: count, dtype: int64


,racecourse_identity,stable_course_identity
0,Aintree,Grand National Course
1,Aintree,Hurdle Course
2,Aintree,Mildmay Course
3,Ascot,National Hunt Course
4,Ascot,Round Course
...,...,...
81,Wincanton,Main Jumps Course (conservative descriptive row)
82,Windsor,Windsor Turf Course
83,Wolverhampton,All-Weather Track (descriptive)
84,Worcester,Main Jumps Course (descriptive)


### Closeout corrections

The closeout audit repaired three venue notebooks before rebuilding this national result. **Carlisle** now has four peer course identities — Flat, Chase, Inner Hurdle and Outer Hurdle — rather than one conservative generic row. **Ayr** remains two peer identities — Flat and Jumps — with its six-furlong Straight Course retained as a lower-level named route/component. **Newmarket** remains Rowley Mile and July Course; official Jockey Club evidence explicitly calls them the two Newmarket racecourses, so Cesarewitch/Beacon terminology is not promoted into extra peer identities.

## Surface as a time-bounded characteristic

In [6]:
surface_history = stable_course_inventory.copy()
surface_history['surface_normalised'] = surface_history['surface'].astype('string').str.strip().str.lower()
surface_history['_valid_from_sort'] = pd.to_datetime(surface_history['valid_from'], errors='coerce')

latest_surface = (
    surface_history
    .sort_values(['racecourse_identity', 'stable_course_identity', '_valid_from_sort'])
    .groupby(['racecourse_identity', 'stable_course_identity'], as_index=False)
    .tail(1)
)

print(latest_surface['surface_normalised'].value_counts(dropna=False))

racecourse_surface_profile = (
    latest_surface.groupby('racecourse_identity')['surface_normalised']
    .agg(lambda s: ' + '.join(sorted(set(s.dropna().astype(str)))))
    .value_counts()
)
print()
print('racecourse surface profiles:')
print(racecourse_surface_profile)

surface_normalised
turf         80
polytrack     3
tapeta        3
Name: count, dtype: int64[pyarrow]

racecourse surface profiles:
surface_normalised
turf                54
polytrack + turf     3
tapeta + turf        2
tapeta               1
Name: count, dtype: int64[pyarrow]


Surface is deliberately not part of stable identity. Newcastle's Flat Track changed from turf to Tapeta and Southwell's all-weather track changed from Fibresand to Tapeta without requiring new persistent track identities. The same modelling principle applies to configuration and operational status.

## Remaining unresolved venue questions

In [7]:
unresolved_tables = []
for path in racecourse_notebooks:
    # This table is optional: two original venue notebooks never created one.
    table = extract_dataframe(path, 'unresolved_questions', required=False)
    if len(table):
        table = table.copy()
        table['source_notebook'] = path.name
        unresolved_tables.append(table)

study03_unresolved = (
    pd.concat(unresolved_tables, ignore_index=True, sort=False)
    if unresolved_tables else pd.DataFrame()
)

print('unresolved records:', len(study03_unresolved))
if len(study03_unresolved):
    print('racecourses with unresolved records:', study03_unresolved['racecourse_identity'].nunique())
    display(study03_unresolved)

unresolved records: 7
racecourses with unresolved records: 6


,racecourse_identity,question,why_unresolved,research_attempted,impact,unresolved_class,verification_status,source_notebook
0,Carlisle,How do the 2015 BHA labels OLD HURDLE and NEW ...,The physical existence of two hurdle courses i...,BHA 2015 distance material; BHA 2025 Inner Hur...,low — historical terminology only; no longer a...,genuine_unresolved,unresolved_after_research,carlisle.ipynb
1,Chelmsford City,What was the exact first date of an official c...,"BHA approval, construction and official physic...",BHA 2018 approval; Chelmsford official Turf Tr...,high — temporal racing-use boundary,genuine_unresolved,unresolved_after_research,chelmsford_city.ipynb
2,Haydock Park,What are the exact physical boundaries and mod...,"The names are supported, but a precise current...",BHA measurement material; current Jockey Club ...,medium — physical hierarchy,genuine_unresolved,unresolved_after_research,haydock_park.ipynb
3,Newcastle,What exact date did the former Flat turf track...,Official sources establish approval/conversion...,BHA conversion material; Study03 first AW obse...,medium — temporal boundary,genuine_unresolved,unresolved_after_research,newcastle.ipynb
4,Nottingham,What are the official names and exact seasonal...,Physical two-course structure is well supporte...,Racing TV; wider web search,medium,genuine_unresolved,unresolved_after_research,nottingham.ipynb
5,Wetherby,What is the exact modern shared-section geomet...,Separate course identities are now authoritati...,Wetherby official course description; BHA 2014...,"low-to-medium — geometry detail, not identity",genuine_unresolved,unresolved_after_research,wetherby.ipynb
6,Wetherby,Which governed alignment is used for NH Flat/b...,BHA fixture material confirms bumpers occur at...,BHA 2021 fixture material; Wetherby official site,low — race-use mapping,genuine_unresolved,unresolved_after_research,wetherby.ipynb


No remaining unresolved item changes the governed national peer-course count. Remaining questions concern bounded detail such as exact temporal boundaries, geometry, operational use or historical terminology and are retained explicitly rather than converted into false precision.

## Sources / provenance

In [8]:
provenance_tables = []
for path in racecourse_notebooks:
    notebook = json.loads(path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        provenance_names = {name for name in names if 'provenance' in name.lower()}
        if not provenance_names:
            continue
        namespace = {'pd': pd}
        try:
            exec(source, namespace)
        except Exception:
            continue
        for name in sorted(provenance_names):
            value = namespace.get(name)
            if isinstance(value, pd.DataFrame) and len(value):
                table = value.copy()
                table['provenance_table'] = name
                table['source_notebook'] = path.name
                provenance_tables.append(table)

study03_provenance = pd.concat(provenance_tables, ignore_index=True, sort=False)
required = ['source_authority', 'source_title', 'source_url', 'accessed_date']
for column in required:
    assert column in study03_provenance.columns
    assert study03_provenance[column].fillna('').astype(str).str.strip().ne('').all(), f'Missing {column}'

study03_sources = (
    study03_provenance[required].drop_duplicates()
    .sort_values(['source_authority', 'source_title', 'source_url']).reset_index(drop=True)
)

print('assertion-level provenance records:', len(study03_provenance))
print('bibliographic source records:', len(study03_sources))
print('unique source URLs:', study03_sources['source_url'].nunique())
print('source notebooks:', study03_provenance['source_notebook'].nunique())
assert study03_provenance['source_notebook'].nunique() == 60
study03_sources

assertion-level provenance records: 1902
bibliographic source records: 165
unique source URLs: 159
source notebooks: 60


,source_authority,source_title,source_url,accessed_date
0,Ascot Racecourse,Contact information – Ascot Shop,https://shop.ascot.com/policies/contact-inform...,2026-08-11
1,Ascot Racecourse,Racing To Zero Summer Mile Family Raceday,https://www.ascot.com/racedays/summer-mile-fam...,2026-08-11
2,Ascot Racecourse,"Venue Hire at Ascot, Berkshire",https://www.ascot.com/venue-hire,2026-08-11
3,At The Races,Ascot Course Guide,https://www.attheraces.com/course-guides/Ascot,2026-08-11
4,At The Races,Bath Course Guide,https://www.attheraces.com/course-guides/Bath,2026-08-11
...,...,...,...,...
160,The Jockey Club,Racecourse Beats The Heatwave To Cut A Dash Ah...,https://www.thejockeyclub.co.uk/carlisle/media...,2026-08-11
161,The Jockey Club,Sandown Park Contact,https://www.thejockeyclub.co.uk/sandown/about/...,2026-08-11
162,The Jockey Club,The History of the Grand National,https://www.thejockeyclub.co.uk/aintree/about/...,2026-08-10
163,The Jockey Club,Treble-seeking Walk In The Mill heads 15 for W...,https://www.thejockeyclub.co.uk/aintree/media/...,2026-08-10


## Conclusion

A British **racecourse** is best treated as the recognised racing venue or institutional identity. It is not necessarily one physical racing course. A racecourse may contain one or several stable **course/track identities**, and those may in turn contain named routes/configurations or characteristics that change through time.

Across Study 03's 60 governed racecourse identities, the repaired venue notebooks produce **90 course/track inventory records**, **86 stable course/track identities**, and **20 racecourses with more than one stable identity**. The 86 is a governed inventory at this modelling level, not a claim that Britain has exactly 86 named physical route distinctions of every possible granularity.

The modelling implication is:

`racecourse -> course/track -> time-bounded characteristics`

A lower route/configuration layer can be added where later analysis needs it. `candidate_course_label` should remain a source-derived classification rather than a governed racecourse or physical track identifier.

**Bottom line: a British racecourse is a venue, not necessarily a single racing course.**

In [9]:
assert source_label_mapping['racecourse_identity'].nunique() == 60
assert source_label_mapping['candidate_course_label'].nunique() == 65
assert len(course_inventory) == 90
assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 20
assert study03_provenance['source_notebook'].nunique() == 60
print('Study 03 national consolidation checks passed.')

Study 03 national consolidation checks passed.
